## Reading in the data

In [ ]:
import pandas as pd
from pathlib import Path

file = Path("../data/crop_data_1965_2025.csv")
df = pd.read_csv(file, encoding='utf-8', sep=',')

df

## Renaming some garbled up column names

In [ ]:
cols = df.columns.tolist()
cols[0] = "Län"
cols[1] = "Gröda"
df.columns = cols
df.columns

## Checking types

In [ ]:
df.dtypes

## Converting numeric columns to floats and creating nulls 

In [ ]:
cols = df.columns[4:]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").astype(float)

## 

## Checking for impossible values

In [ ]:
df[(df[cols] < 0).any(axis=1)]

## Checking unique values

In [ ]:
unique_crops = df["Gröda"].unique().tolist()
print(unique_crops)

In [ ]:
unique_regions = df["Län"].unique().tolist()
print(unique_regions)

In [ ]:
unique_variables = df["Variabel"].unique().tolist()
print(unique_variables)

In [ ]:
unique_records = df["Tabelluppgift"].unique().tolist()
print(unique_records)

## TODO: Convert the strange values

In [ ]:
fixes = {
    "H�stvete": "Höstvete",
    "V�rvete": "Vårvete",
    "R�g": "Råg",
    "H�stkorn": "Höstkorn",
    "V�rkorn": "Vårkorn",
    "Havre": "Havre",
    "R�gvete": "Rågvete",
    "H�str�gvete": "Höstrågvete",
    "V�rr�gvete": "Vårrågvete",
    "Blands�d": "Blandsäd",
    "Majs": "Majs",
    "Spannm�l totalt": "Spannmål totalt",
    "�rter": "Ärter",
    "�kerb�nor": "Åkerbönor",
    "H�straps": "Höstraps",
    "V�rraps": "Vårraps",
    "H�strybs": "Höstrybs",
    "V�rrybs": "Vårrybs",
    "Str�s�desgr�dor (exkl. majs) till gr�nfoder": "Stråsädesgrödor (exkl. majs) till grönfoder",
    "Majs till gr�nfoder": "Majs till grönfoder",
    "Andra ett�riga gr�dor till gr�nfoder": "Andra ettåriga grödor till grönfoder",
    "Oljelin": "Oljelin",
    "Matpotatis": "Matpotatis",
    "Potatis f�r st�rkelse": "Potatis för stärkelse",
    "Sockerbetor": "Sockerbetor",
    "Sl�ttervall, f�rsta sk�rd": "Slåttervall, första skörd",
    "Sl�ttervall, �terv�xt": "Slåttervall, återväxt",
    "Sl�ttervall, total vallsk�rd": "Slåttervall, total vallskörd",
}


df["Gröda"] = df["Gröda"].replace(fixes)

df["Gröda"].unique().tolist()

In [ ]:
lan_short = {
    "00": "Riket",
    "01": "Stockholm",
    "03": "Uppsala",
    "04": "Södermanland",
    "05": "Östergötland",
    "06": "Jönköping",
    "07": "Kronoberg",
    "08": "Kalmar",
    "09": "Gotland",
    "10": "Blekinge",
    "11": "Kristianstad",
    "12": "Skåne",
    "122": "Malmöhus",
    "13": "Halland",
    "14": "Västra Götaland",
    "141": "Göteborg och Bohus",
    "15": "Älvsborg",
    "16": "Skaraborg",
    "17": "Värmland",
    "18": "Örebro",
    "19": "Västmanland",
    "20": "Dalarna",
    "21": "Gävleborg",
    "22": "Västernorrland",
    "23": "Jämtland",
    "24": "Västerbotten",
    "25": "Norrbotten",
}

df["Län_kod"] = df["Län"].str.extract(r"^(\d+)", expand=False)
df["Län"] = df["Län_kod"].map(lan_short)

df.head()

In [ ]:
df["Variabel"] = df["Variabel"].replace({"Totalsk�rd, ton": "Totalskörd, ton"})

In [ ]:
df = df.drop(columns=["Län_kod", "Tabelluppgift"])

In [ ]:
df.columns

In [ ]:
df.columns = ["Län", "Gröda", "Variabel"] + list(df.columns[3:])
df["Variabel"] = df["Variabel"].str.startswith("Areal").map({True: "area_ha", False: "harvest_ton"})

tidy = df.pivot(index=["Län", "Gröda"], columns="Variabel")

tidy.head()

In [ ]:
long = df.melt(id_vars=["Län", "Gröda", "Variabel"], var_name="År", value_name="value")
tidy = long.pivot(index=["Län", "Gröda", "År"], columns="Variabel", values="value").reset_index()
tidy.columns.name = None   # removes the leftover 'Variabel' label above the columns

tidy

In [ ]:
duplicates = tidy.duplicated(
    subset=['Län', 'Gröda', 'År']
).sum()

duplicates

In [ ]:
valid = tidy[
    tidy['area_ha'].notna() &
    tidy['harvest_ton'].notna() &
    (tidy['area_ha'] > 0) &
    (tidy['harvest_ton'] > 0)
].copy()

valid['yield_ton_ha'] = (
    valid['harvest_ton'] / valid['area_ha']
)

valid

In [ ]:
coverage = (
    valid.groupby('Gröda')
    .agg(
        observations=('yield_ton_ha', 'count'),
        counties=('Län', 'nunique'),
        years=('År', 'nunique'),
        first_year=('År', 'min'),
        last_year=('År', 'max')
    )
    .sort_values('observations', ascending=False)
)
coverage

In [ ]:
(valid['area_ha'] <= 0).sum(), (valid['harvest_ton'] <= 0).sum()   # should both be 0
valid[["area_ha", "harvest_ton", "yield_ton_ha"]].isna().sum()

In [ ]:
d = valid.rename(columns={"Län": "lan", "Gröda": "crop", "År": "year"}).copy()
d["yield_ton_ha"] = d.harvest_ton / d.area_ha   # keep column consistent with its inputs

# Which crops are in the data, and how much?
print(d.groupby("crop").agg(
    n_obs=("yield_ton_ha", "size"),
    n_years=("year", "nunique"),
    n_lan=("lan", "nunique"),
    median_yield=("yield_ton_ha", "median"),
    max_yield=("yield_ton_ha", "max"),
).sort_values("median_yield", ascending=False))

In [ ]:
# 1. Inspect the impossible slåttervall values
bad = d[(d.crop == "Slåttervall, första skörd") & (d.yield_ton_ha > 15)]
print(bad)
d = d.drop(bad.index)          # or fix the units if you can trace the cause

# 2. Define groups
groups = {
    "cereals":  ["Höstvete", "Höstkorn", "Höstrågvete", "Råg", "Rågvete",
                 "Vårvete", "Vårkorn", "Havre", "Blandsäd"],
    "oilseeds": ["Höstraps", "Vårraps", "Höstrybs", "Vårrybs", "Oljelin"],
    "pulses":   ["Åkerbönor", "Ärter"],
    "roots":    ["Matpotatis", "Potatis för stärkelse", "Sockerbetor"],
}

def summarize(crops):
    g = d[d.crop.isin(crops)]
    out = g.groupby("crop").agg(harvest=("harvest_ton", "sum"),
                                area=("area_ha", "sum"),
                                sd=("yield_ton_ha", "std"),
                                mean=("yield_ton_ha", "mean"),
                                n_lan=("lan", "nunique"),
                                n_years=("year", "nunique"))
    out["wtd_yield"] = out.harvest / out.area
    out["cv"] = out.sd / out["mean"]
    return out.sort_values("wtd_yield", ascending=False).round(2)

for name, crops in groups.items():
    print(f"\n== {name} ==")
    print(summarize(crops))

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

def crop_effects(crops, ref, data=d):
    g = data[data.crop.isin(crops)].copy()
    m = smf.wls(
        f"np.log(yield_ton_ha) ~ C(crop, Treatment('{ref}')) + C(lan) + C(year)",
        data=g, weights=g.area_ha,
    ).fit(cov_type="cluster",
          cov_kwds={"groups": g.lan.astype("category").cat.codes})
    fx = m.params.filter(like="crop")
    fx.index = [s.split("[T.")[1].rstrip("]") for s in fx.index]
    ci = m.conf_int().loc[m.params.filter(like="crop").index]
    ci.index = fx.index
    res = pd.DataFrame({"pct_vs_ref": (np.exp(fx) - 1) * 100,
                        "ci_low": (np.exp(ci[0]) - 1) * 100,
                        "ci_high": (np.exp(ci[1]) - 1) * 100})
    return res.sort_values("pct_vs_ref", ascending=False).round(1)

print(crop_effects(groups["cereals"], ref="Höstvete"))
print(crop_effects(groups["oilseeds"], ref="Höstraps"))

In [ ]:
d["year"] = pd.to_numeric(d["year"], errors="raise").astype(int)

# 1. Does the ranking hold in recent years only?
print(crop_effects(groups["cereals"], ref="Höstvete", data=d[d.year >= 2015]))

# 2. Does the crop effect vary by region? (example: crop x county for the top crops)
top = d[d.crop.isin(["Höstvete", "Vårkorn", "Havre"])]
print(top.pivot_table(index="lan", columns="crop", values="yield_ton_ha", aggfunc="mean").round(2))

In [28]:
tidy.to_csv("../data/crop_data_clean.csv", index=False, encoding="utf-8")